# Your DI Validator notebook
Run each code cell with **Shift+Enter**. This notebook reads the current local workspace. The API preserves native measurements and their provenance; it does not return downsampled chart data.

Edits are saved in this browser. Use **File → Save and Export Notebook As → Notebook** (or download from the file browser) for a portable `.ipynb` copy.

In [ ]:
import pandas as pd
from di_data import DIClient

di = DIClient()
datasets = await di.datasets()
pd.DataFrame(datasets)[['id', 'name', 'format', 'asset_level', 'assets', 'start', 'end']]

## Inspect an hourly transformer or meter profile
Choose a dataset and asset below. Change `start`, `end`, and `limit` to select a window. Data remains in its declared unit; negative net energy is preserved.

In [ ]:
dataset = next(d for d in datasets if d['format'] == 'wide_ami')
assets = await di.assets(dataset['id'])
quality = await di.quality(dataset['id'])
observed = {r['asset_id']: r.get('observed', 0) for r in quality['assets']}
assets.sort(key=lambda a: observed.get(a['id'], 0), reverse=True)
pd.DataFrame(assets).head()

In [ ]:
page = await di.readings(dataset['id'], asset_ids=[assets[0]['id']], limit=10000)
frame = di.to_frame(page)
print('Units:', frame.attrs['channels'])
print('More data:', page['next_cursor'] is not None)
frame.head()

In [ ]:
import matplotlib.pyplot as plt
profile = frame.set_index('timestamp')['value']
ax = profile.plot(style='.', figsize=(12, 4), markersize=2)
ax.set_title(assets[0]['source_id'])
ax.set_ylabel(dataset['channels'][0]['unit'])
plt.show()

## Quality and label evidence
Missing readings remain missing. Invalid/ambiguous clock rows are excluded from time-aligned analysis and retained in the audit. Registry nonmatches are not confirmed negatives. Keep circuit and asset level when joining labels.

In [ ]:
quality = await di.quality(dataset['id'])
quality['summary']

In [ ]:
registries = await di.registries()
if registries:
    registry = await di.registry(registries[0]['id'])
    display(registry.head())
manual_label_revisions = await di.labels()
print('Manual label revisions:', len(manual_label_revisions))

## Larger analyses
Iterate bounded pages instead of concatenating a full AMI cohort or 1 kHz recording. Each page carries metadata in `chunk.attrs`. Use the worker experiment workflows for the application's frozen, leakage-safe validation.

In [ ]:
observed = 0
energy = 0.0
async for chunk in di.iter_frames(dataset['id'], asset_ids=[assets[0]['id']], limit=5000):
    observed += chunk['value'].count()
    energy += chunk['value'].sum()
print({'observed_intervals': int(observed), 'net_energy': float(energy), 'unit': dataset['channels'][0]['unit']})